# FastText

This notebook has two parts:
1. **Load a pretrained FastText model** — requires the dataset file `wiki-news-300d-1M.vec` (not included in this repository, see the download instructions below)
2. **Train your own FastText model** — requires no dataset; the training sentences are defined inside the notebook

> **Note on the dataset format:** The `.vec` file is a plain-text file. Its first line is `1000000 300` (number of words, vector dimensions), and every following line contains a word followed by its 300 vector values. That is why the loading code uses `binary=False` — gensim can read this text format directly, no conversion is needed.

## Setup (one time only)
In a terminal / Anaconda Prompt:
```bash
pip install gensim jupyter
jupyter notebook FastText.ipynb
```
Then run every cell from top to bottom using `Shift + Enter`.

## Download the pretrained dataset (required for Part 1)
- Official source (Facebook AI): https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip (~1.1 GB zip, ~5 GB after extraction)
- Alternatively, search for `wiki-news-300d-1M.vec` on Kaggle

Extract the zip and place **`wiki-news-300d-1M.vec` in the same folder as this notebook**, otherwise you will get a `FileNotFoundError`.
If your machine has limited RAM (8 GB or less), keep `limit=50000` in the loading cell below — the full 1M-word model is not required for this lab.

In [ ]:
import os
from gensim.models import KeyedVectors

model_path = "wiki-news-300d-1M.vec"   # the .vec file is in plain-text format

if not os.path.exists(model_path):
    fasttext_model = None
    print("Dataset not found:", model_path)
    print("Download it (see the instructions above) and place it in the same folder as this notebook.")
    print("Part 2 (training your own model) below will still run without the dataset.")
else:
    # limit=50000 loads only the first 50,000 words (faster, uses less RAM).
    # Set limit=None to load the full vocabulary.
    fasttext_model = KeyedVectors.load_word2vec_format(model_path, binary=False, limit=50000)
    print("Vocabulary size:", len(fasttext_model))
    print("Vector dimensions:", fasttext_model.vector_size)

In [ ]:
if fasttext_model is None:
    print("Pretrained model is not loaded - skipping this cell")
else:
    # Get the vector of a word
    word = "computer"

    vector = fasttext_model[word]

    print("Word:", word)
    print("Vector shape:", vector.shape)
    print("First 10 values:", vector[:10])

In [ ]:
if fasttext_model is None:
    print("Pretrained model is not loaded - skipping this cell")
else:
    # Find similar words
    similar_words = fasttext_model.most_similar(
        "computer",
        topn=5
    )

    for word, score in similar_words:
        print(word, round(score, 4))

In [ ]:
if fasttext_model is None:
    print("Pretrained model is not loaded - skipping this cell")
else:
    # Compare word similarities
    pairs = [
        ("king", "queen"),
        ("computer", "laptop"),
        ("computer", "banana")
    ]

    for word1, word2 in pairs:
        score = fasttext_model.similarity(word1, word2)
        print(word1, "-", word2, ":", round(score, 4))

In [ ]:
# Test a known word and an unseen word
# NOTE (fix): this cell previously used an undefined variable `pretrained_ft`
# with a `get_word_vector()` method, which raised a NameError.
# A KeyedVectors model loaded from a .vec/.txt file cannot produce a vector
# for an out-of-vocabulary (unseen) word, because the .vec file does not store
# subword (character n-gram) information. Unseen-word vectors require either
# the .bin pretrained model or a model trained with FastText itself (see Part 2).
if fasttext_model is None:
    print("Pretrained model is not loaded - skipping this cell")
else:
    print("Is 'playing' in the vocabulary:", "playing" in fasttext_model.key_to_index)
    print("Vector shape:", fasttext_model["playing"].shape)
    try:
        print(fasttext_model["playfullyxyz"].shape)
    except KeyError:
        print("'playfullyxyz' is not in the vocabulary, so the .vec model cannot return a vector for it.")

Pretrained vectors have 300 dimensions. An unseen-word vector is constructed from character subword information, so it requires a `.bin` model or a self-trained FastText model — a `.vec/.txt` file alone is not sufficient. It also does not mean that the model has learned the actual meaning of an arbitrary invented word.

## Train your own FastText model

In [ ]:
from gensim.models import FastText
from gensim.utils import simple_preprocess

sentences = [
    "machine learning is useful",
    "deep learning is a part of machine learning",
    "word embeddings represent words as vectors",
    "fasttext learns word representations from subwords",
    "word2vec learns useful word embeddings",
    "natural language processing uses word embeddings",
    "playing played player plays",
    "machine learning models learn patterns from data",
    "deep neural networks learn representations",
    "fasttext represents words using character ngrams"
]

tokenized_sentences = [
    simple_preprocess(sentence)
    for sentence in sentences
]

for sentence in tokenized_sentences[:3]:
    print(sentence)

In [ ]:
#Train the FastText model
ft_model = FastText(
    sentences=tokenized_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    min_n=3,
    max_n=6,
    epochs=100,
    seed=42
)

print("Vocabulary size:", len(ft_model.wv))
print("Vector dimensions:", ft_model.wv.vector_size)

In [ ]:
#Retrieve a trained word vector
word = "learning"

vector = ft_model.wv[word]

print("Word:", word)
print("Vector shape:", vector.shape)
print("First 10 values:", vector[:10])

In [ ]:
#Find similar words
print(
    ft_model.wv.most_similar(
        "learning",
        topn=5
    )
)

In [ ]:
known_word = "playing"
unseen_word = "playfullyxyz"

print("Known word in vocabulary:",
      known_word in ft_model.wv.key_to_index)

print("Unseen word in vocabulary:",
      unseen_word in ft_model.wv.key_to_index)

print("Known vector shape:",
      ft_model.wv[known_word].shape)

print("Unseen vector shape:",
      ft_model.wv[unseen_word].shape)

In [ ]:
#Save and reload your own model

ft_model.save("my_fasttext.model")

loaded_model = FastText.load("my_fasttext.model")

print(loaded_model.wv["learning"][:10])
print(loaded_model.wv["playfullyxyz"][:10])

---
### Expected output
- After training in Part 2: `Vocabulary size: 37`, `Vector dimensions: 100`
- `ft_model.wv['learning']` returns a vector of shape `(100,)`
- The Part 2 model can also return a vector for the unseen word `playfullyxyz` (shape `(100,)`) because it uses subword information — this is the main feature of FastText
- At the end, the trained model is saved as `my_fasttext.model` in the same folder